<a href="https://colab.research.google.com/github/440g/painkiller/blob/Model-evaluatioin/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model training and validation
* Selection of multiple machine learning algorithms.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [20]:
import pandas as pd

X_train = pd.read_csv('../datasets/X_train.csv')
#파일 맨 앞에 의미없는 인덱스 값이 존재해 제거
X_train = X_train.drop(columns=['Unnamed: 0'])
y_train = pd.read_csv('../datasets/y_train.csv')
X_val = pd.read_csv('../datasets/X_val.csv')
X_val = X_val.drop(columns=['Unnamed: 0'])
y_val = pd.read_csv('../datasets/y_val.csv')

In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.model_selection import KFold, GridSearchCV

In [7]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = "f1"

models = {}

### 1. Decision Tree

In [8]:
model = DecisionTreeClassifier(random_state=42)


param_grid = {
    "max_depth": [5, 10, 20],
    "min_samples_split": [2, 10, 20],
    "ccp_alpha": [0.0, 0.01],
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True, n_jobs=-1)
grid_search.fit(X_train, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["Decision Tree"] = grid_search.best_estimator_

Best parameters:  {'ccp_alpha': 0.0, 'max_depth': 5, 'min_samples_split': 2}
Best CV score: 0.635988


### 2. Bagging

In [9]:
model = BaggingClassifier(estimator=DecisionTreeClassifier(),
                         n_jobs=-1,
                         random_state=42)


param_grid = {
    "n_estimators": [25, 50]
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True, n_jobs=-1)
grid_search.fit(X_train, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["Bagging"] = grid_search.best_estimator_

Best parameters:  {'n_estimators': 50}
Best CV score: 0.624973


### 3. Random Forest

In [10]:
model = RandomForestClassifier(n_jobs=-1, random_state=42)


param_grid = {
    "n_estimators": [25, 50],
    "max_features": ["sqrt", "log2"],
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True, n_jobs=-1)
grid_search.fit(X_train, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["Random Forest"] = grid_search.best_estimator_

Best parameters:  {'max_features': 'log2', 'n_estimators': 50}
Best CV score: 0.631532


### 4. AdaBoost

In [11]:
model = AdaBoostClassifier(random_state=42)


param_grid = {
    "estimator": [DecisionTreeClassifier(max_depth=3), DecisionTreeClassifier(max_depth=6)],
    "learning_rate": [0.1, 1.0],
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True, n_jobs=-1)
grid_search.fit(X_train, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["AdaBoost"] = grid_search.best_estimator_

Best parameters:  {'estimator': DecisionTreeClassifier(max_depth=6), 'learning_rate': 0.1}
Best CV score: 0.648480


### 5. Gradient Boosting

In [12]:
model = GradientBoostingClassifier(
                                  random_state=42)

param_grid = {
    "max_depth": [3, 6],
    "learning_rate": [0.0, 0.1],
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True, n_jobs=-1)
grid_search.fit(X_train, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["Gradient Boosting"] = grid_search.best_estimator_

Best parameters:  {'learning_rate': 0.1, 'max_depth': 3}
Best CV score: 0.652684


### 6. XG Boost

In [13]:
model = XGBClassifier(learning_rate=0.1,
                     n_jobs=-1,
                     random_state=42)

param_grid = {
    "n_estimators": [25, 50],
    "reg_alpha": [0, 0.1],
    "reg_lambda": [0, 0.1],
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True)
grid_search.fit(X_train, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["XGBoost"] = grid_search.best_estimator_

Best parameters:  {'n_estimators': 50, 'reg_alpha': 0, 'reg_lambda': 0.1}
Best CV score: 0.654553


### 7. Light GBM

In [14]:
model = LGBMClassifier(data_sample_strategy="goss",
                      top_rate=0.2,
                      other_rate=0.1,
                      force_col_wise=True,
                      verbosity=0,
                      n_jobs=-1,
                      random_state=42)

#Light GBM 모델이 특수기호가 있으면 읽지 못해 없애는 과정이 필요함
X_train_lgbm = X_train.copy()
X_val_lgbm = X_val.copy()
X_train_lgbm.columns = X_train_lgbm.columns.str.replace(r"[^\w]", "_", regex = True)
X_val_lgbm.columns = X_val_lgbm.columns.str.replace(r"[^\w]", "_", regex = True)

param_grid = {
    "n_estimators": [25, 50],
    "reg_alpha": [0, 0.1],
    "reg_lambda": [0, 0.1],
    "enable_bundle": [True, False]
}

grid_search = GridSearchCV(model, param_grid, cv=kf, scoring=scoring, refit=True)
grid_search.fit(X_train_lgbm, y_train['y'])

print("Best parameters: ", grid_search.best_params_)
print("Best CV score: {:.6f}".format(grid_search.best_score_))

models["LightGBM"] = grid_search.best_estimator_

Best parameters:  {'enable_bundle': True, 'n_estimators': 25, 'reg_alpha': 0.1, 'reg_lambda': 0}
Best CV score: 0.644324


In [15]:
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

for _name, _model in models.items():
    y_pred = _model.predict(X_val)

    f1 = f1_score (y_val["y"], y_pred)
    accuracy = accuracy_score(y_val["y"], y_pred)
    AUC = roc_auc_score (y_val["y"], y_pred)
    Main_Evaluation = (f1 + accuracy + AUC)/3
    print("{:>17}: f1 = {:.4f} | accuracy = {:.4f} | AUC = {:.4f} | Main Evaluation = {:.4f}".format(
        _name, f1, accuracy, AUC, Main_Evaluation))



    Decision Tree: f1 = 0.6286 | accuracy = 0.6087 | AUC = 0.6095 | Main Evaluation = 0.6156
          Bagging: f1 = 0.6275 | accuracy = 0.6391 | AUC = 0.6388 | Main Evaluation = 0.6351
    Random Forest: f1 = 0.6347 | accuracy = 0.6488 | AUC = 0.6484 | Main Evaluation = 0.6440
         AdaBoost: f1 = 0.6395 | accuracy = 0.6543 | AUC = 0.6539 | Main Evaluation = 0.6492
Gradient Boosting: f1 = 0.6534 | accuracy = 0.6627 | AUC = 0.6624 | Main Evaluation = 0.6595
          XGBoost: f1 = 0.6422 | accuracy = 0.6499 | AUC = 0.6498 | Main Evaluation = 0.6473
         LightGBM: f1 = 0.6289 | accuracy = 0.6425 | AUC = 0.6422 | Main Evaluation = 0.6379


In [26]:
X_test = pd.read_csv('../datasets/X_test.csv').drop(columns=['Unnamed: 0', 'id'])
id = pd.read_csv('../datasets/X_test.csv')['id']

In [ ]:
best_model = models["Gradient Boosting"]
predictions = best_model.predict(X_test)

pd.DataFrame({'id': id, 'prediction': predictions}).to_csv('../datasets/prediction.csv', index=False)